# Parte 1

In [2]:
import numpy as np
import pandas as pd
import matplotlib as plt
import funcoes_aux_proj_apr_supervisionada

dados = pd.read_pickle('Xsup.pkl')
dados

,Skeleton_Sequence,Exercise_Id,Patient_ID,Compensação_Ombro,Compensação_Tronco,Compensação_Cabeça
0,"[[-0.025484836, -0.572286367, -0.011180799, -0...",E2,18,0,1,1
1,"[[0.1457298099994659, -0.177351892, 0.17020170...",E4,17,1,1,1
2,"[[0.019437663, -0.542804241, 0.032009996473789...",E5,15,0,0,1
3,"[[-0.000282567, -0.572014868, 0.014581724, -0....",E1,2,1,1,1
4,"[[-0.067052722, -0.077053629, -0.050083861, -0...",E4,8,1,0,1
...,...,...,...,...,...,...
1254,"[[0.062138677, -0.535513282, 0.077235512, -0.5...",E5,3,1,0,1
1255,"[[0.061319716, -0.557576954, 0.074186742, -0.5...",E5,11,1,0,1
1256,"[[-0.009194537, -0.536099017, 0.00539052, -0.5...",E3,1,0,1,0
1257,"[[-0.045662187, -0.582765043, -0.031464174, -0...",E2,11,0,1,1


## R1a) - Função de regressão linear

In [3]:
import numpy as np 

def regressao_linear (X, Y):
    X = np.array(X)
    Y = np.array(Y)
    
    theta, residuals, rank, s = np.linalg.lstsq(X, Y, rcond=None)
    
    beta_0 = theta[0]
    beta = theta[1:]
    
    return beta, beta_0

## R1b)

In [4]:
sequencias = dados[(dados["Exercise_Id"] == "E5") & (dados["Patient_ID"] == 4)]

n = len(sequencias)
n_treino = int(n*0.8)

treino = sequencias.iloc[:n_treino]
teste = sequencias.iloc[n_treino:]

print("Total de Sequências:", n)
print("Sequências de Treino:", len(treino))
print("Sequências de Teste:", len(teste))


Total de Sequências: 10
Sequências de Treino: 8
Sequências de Teste: 2


## R1c)

In [5]:
X_treino_lista = []
Y_treino_lista = []

Nome_coluna = "Skeleton_Sequence"

for _, row in treino.iterrows():
    seq = np.array(row[Nome_coluna]) if Nome_coluna in treino.columns else np.array(row)
    X_treino_lista.append(seq[:-1, :])
    Y_treino_lista.append(seq[1:, :])
    
X_treino_raw = np.vstack(X_treino_lista)
Y_treino_raw = np.vstack(Y_treino_lista)

X_treino = np.column_stack((np.ones(X_treino_raw.shape[0]), X_treino_raw))

print("Dimensão de X_treino:", X_treino.shape)
print("Dimensão de Y_treino:", Y_treino_raw.shape)
    

Dimensão de X_treino: (279, 67)
Dimensão de Y_treino: (279, 66)


## R1d)

In [6]:
X_teste_lista = []
Y_teste_lista = []

NOME_COLUNA = "Skeleton_Sequence" 

for _, row in teste.iterrows():
    seq = np.array(row[NOME_COLUNA])
    X_teste_lista.append(seq[:-1, :])
    Y_teste_lista.append(seq[1:, :])
    
X_teste_raw = np.vstack(X_teste_lista)
Y_teste_raw = np.vstack(Y_teste_lista)

# Adicionar obrigatoriamente a coluna de 1s para o beta_0 no conjunto de teste também
X_teste = np.column_stack((np.ones(X_teste_raw.shape[0]), X_teste_raw))

print("Dimensão de X_teste:", X_teste.shape)
print("Dimensão de Y_teste:", Y_teste_raw.shape)


Dimensão de X_teste: (67, 67)
Dimensão de Y_teste: (67, 66)


## R1e)

In [7]:
def calcular_metricas(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    sse = np.sum((y_true - y_pred) ** 2)
    
    y_mean = np.mean(y_true, axis=0)
    sst = np.sum((y_true - y_mean) ** 2)
    
    r2 = 1 - (sse / sst)
    
    return sse, r2

## R2a)

In [8]:
beta, beta_0 = regressao_linear(X_treino, Y_treino_raw)

print("Dimensões de beta:", beta.shape)
print("Dimensões de beta_0:", beta_0.shape)


Dimensões de beta: (66, 66)
Dimensões de beta_0: (66,)


## R2b)

In [9]:
Y_treino_pred = X_treino_raw @ beta + beta_0

sse_treino, r2_treino = calcular_metricas(Y_treino_raw, Y_treino_pred)

print(f"SSE Global: {sse_treino:.4f}")
print(f"R^2 Global: {r2_treino:.4f}")


SSE Global: 1.4345
R^2 Global: 0.9666


/var/folders/d7/36p1n7fd4c7fdwkb6r3rvbkw0000gn/T/ipykernel_98074/2144122150.py:1: RuntimeWarning: divide by zero encountered in matmul
  Y_treino_pred = X_treino_raw @ beta + beta_0
/var/folders/d7/36p1n7fd4c7fdwkb6r3rvbkw0000gn/T/ipykernel_98074/2144122150.py:1: RuntimeWarning: overflow encountered in matmul
  Y_treino_pred = X_treino_raw @ beta + beta_0
/var/folders/d7/36p1n7fd4c7fdwkb6r3rvbkw0000gn/T/ipykernel_98074/2144122150.py:1: RuntimeWarning: invalid value encountered in matmul
  Y_treino_pred = X_treino_raw @ beta + beta_0


## R2c)

In [10]:
Y_teste_pred = X_teste_raw @ beta + beta_0

sse_teste, r2_teste = calcular_metricas(Y_teste_raw, Y_teste_pred)

print(f"SSE Global: {sse_teste:.4f}")
print(f"R^2 Global: {r2_teste:.4f}")

erros_por_keypoint = np.sum((Y_teste_raw - Y_teste_pred) ** 2, axis=0)
top_erros_idx = np.argsort(erros_por_keypoint)[::-1][:5]

print(f"\nOs 5 índices (0-65) com maior erro (SSE) acumulado no teste: {top_erros_idx}")


SSE Global: 0.7344
R^2 Global: 0.9291

Os 5 índices (0-65) com maior erro (SSE) acumulado no teste: [63 59 53 55 51]


/var/folders/d7/36p1n7fd4c7fdwkb6r3rvbkw0000gn/T/ipykernel_98074/323876059.py:1: RuntimeWarning: divide by zero encountered in matmul
  Y_teste_pred = X_teste_raw @ beta + beta_0
/var/folders/d7/36p1n7fd4c7fdwkb6r3rvbkw0000gn/T/ipykernel_98074/323876059.py:1: RuntimeWarning: overflow encountered in matmul
  Y_teste_pred = X_teste_raw @ beta + beta_0
/var/folders/d7/36p1n7fd4c7fdwkb6r3rvbkw0000gn/T/ipykernel_98074/323876059.py:1: RuntimeWarning: invalid value encountered in matmul
  Y_teste_pred = X_teste_raw @ beta + beta_0


## R2d)

In [12]:
from funcoes_aux_proj_apr_supervisionada import visualize_sequences
from IPython.display import display

NOME_COLUNA = "Skeleton_Sequence" 
seq_verdadeira = np.array(teste.iloc[0][NOME_COLUNA])

X_seq_teste = seq_verdadeira[:-1, :]
Y_seq_teste_pred = X_seq_teste @ beta + beta_0

anim_true = visualize_sequences(seq_verdadeira[1:], title="Sequência Verdadeira (E5)")

anim_pred = visualize_sequences(Y_seq_teste_pred, title="Sequência Prevista (E5)")

display(anim_true)
display(anim_pred)

/var/folders/d7/36p1n7fd4c7fdwkb6r3rvbkw0000gn/T/ipykernel_98074/607330769.py:8: RuntimeWarning: divide by zero encountered in matmul
  Y_seq_teste_pred = X_seq_teste @ beta + beta_0
/var/folders/d7/36p1n7fd4c7fdwkb6r3rvbkw0000gn/T/ipykernel_98074/607330769.py:8: RuntimeWarning: overflow encountered in matmul
  Y_seq_teste_pred = X_seq_teste @ beta + beta_0
/var/folders/d7/36p1n7fd4c7fdwkb6r3rvbkw0000gn/T/ipykernel_98074/607330769.py:8: RuntimeWarning: invalid value encountered in matmul
  Y_seq_teste_pred = X_seq_teste @ beta + beta_0


Total Frames to Animate: 34
Total Frames to Animate: 34


# Parte II

## R3a)